# BigQuery Explorer

Query BigQuery tables and return results as pandas DataFrames.

In [1]:
# Install dependencies (run once)
%pip install google-cloud-bigquery pandas db-dtypes pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT = "realtime-headway-prediction"
client = bigquery.Client(project=PROJECT)
print(f"Connected to project: {client.project}")

Connected to project: realtime-headway-prediction


In [3]:
query = """
select
*
from `headway_prediction.clean`
where route_id in ('A', 'C', 'E')
  and direction = 'S'
"""

df = client.query(query).to_dataframe()
df.head()

,trip_uid,stop_id,track,trip_date,route_id,direction,arrival_time
0,1747931220_A..S58R,A02S,A3,2025-05-22 16:27:00+00:00,A,S,2025-05-22 20:27:00+00:00
1,1755894345_A..S58R,A02S,A3,2025-08-22 20:25:45+00:00,A,S,2025-08-23 00:27:03+00:00
2,1767783300_A..S58R,A02S,A3,2026-01-07 10:55:00+00:00,A,S,2026-01-07 15:55:46+00:00
3,1754472420_A..S58R,A02S,A3,2025-08-06 09:27:00+00:00,A,S,2025-08-06 13:27:12+00:00
4,1766133619_A..S58R,A02S,A3,2025-12-19 08:40:19+00:00,A,S,2025-12-19 13:40:57+00:00


In [4]:
# Quick summary of the returned data
print(f"Shape: {df.shape}")
df.info()


Shape: (4757706, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4757706 entries, 0 to 4757705
Data columns (total 7 columns):
 #   Column        Dtype              
---  ------        -----              
 0   trip_uid      object             
 1   stop_id       object             
 2   track         object             
 3   trip_date     datetime64[us, UTC]
 4   route_id      object             
 5   direction     object             
 6   arrival_time  datetime64[us, UTC]
dtypes: datetime64[us, UTC](2), object(5)
memory usage: 254.1+ MB


### Validate Stops for Valid Routes

In [5]:
stops = pd.read_csv('../local_artifacts/raw_data/stops.txt')

### Handle missing data and define node order

In [6]:
# helper function to find routes with origin and terminal
# grouped by number of stops to help build topography in graph database

def route_finder(route, origin, terminal):
    df_a = df.loc[df['route_id']== route]

    # find trips with both origin and terminal
    trips_with_origin = (set(df_a.loc[df_a['stop_id']== origin, 'trip_uid']))
    trips_with_terminal = (set(df_a.loc[df_a['stop_id']== terminal, 'trip_uid']))

    matching_trips = trips_with_origin & trips_with_terminal # intersection

    # filter those trip_uids
    result = (df_a.loc[df_a['trip_uid'].isin(matching_trips)]
                 .groupby('trip_uid')['trip_uid']
                 .count()
                 .to_frame('counts')
                 .reset_index())

    return result

In [7]:
stops.loc[stops.stop_name.str.contains('World')]

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
858,E01,World Trade Center,40.712582,-74.009781,1.0,NaN
859,E01N,World Trade Center,40.712582,-74.009781,NaN,E01
860,E01S,World Trade Center,40.712582,-74.009781,NaN,E01


In [8]:
route, origin, terminal = 'E', 'G05S', 'E01S'

route_finder(route, origin, terminal)


,trip_uid,counts
0,1739769240_E..S04R,32
1,1739770380_E..S69R,22
2,1739771460_E..S69R,22
3,1739772300_E..S69R,22
4,1739773020_E..S69R,22
...,...,...
56491,1771299090_E..S04R,32
56492,1771300290_E..S04R,32
56493,1771301490_E..S04R,32
56494,1771302690_E..S04R,32


In [9]:
a_far_rockaway =  (
                        df.loc[df.trip_uid=='1747632300_A..S58R']
                        .sort_values(by=['arrival_time'], ascending=True)
                        #.merge(stops, on=['stop_id'], how='left')
                        ['stop_id']
                        .tolist()
                    )
a_far_rockaway

['A02S',
 'A03S',
 'A05S',
 'A06S',
 'A07S',
 'A09S',
 'A12S',
 'A15S',
 'A24S',
 'A27S',
 'A28S',
 'A31S',
 'A32S',
 'A34S',
 'A36S',
 'A38S',
 'A40S',
 'A41S',
 'A42S',
 'A46S',
 'A48S',
 'A51S',
 'A55S',
 'A57S',
 'A59S',
 'A60S',
 'A61S',
 'H02S',
 'H03S',
 'H04S',
 'H06S',
 'H07S',
 'H08S',
 'H09S',
 'H10S',
 'H11S']

In [10]:
a_lefferts =    (
                        df.loc[df.trip_uid=='1739773170_A..S57R']
                        .sort_values(by=['arrival_time'], ascending=True)
                        #.merge(stops, on=['stop_id'], how='left')
                        ['stop_id']
                        .tolist()
                        )
a_lefferts

['A02S',
 'A03S',
 'A05S',
 'A06S',
 'A07S',
 'A09S',
 'A12S',
 'A15S',
 'A24S',
 'A27S',
 'A28S',
 'A31S',
 'A32S',
 'A34S',
 'A36S',
 'A38S',
 'A40S',
 'A41S',
 'A42S',
 'A46S',
 'A48S',
 'A51S',
 'A55S',
 'A57S',
 'A59S',
 'A60S',
 'A61S',
 'A63S',
 'A64S',
 'A65S']

In [11]:
a_rockaway_park =    (
                        df.loc[df.trip_uid=='1754688720_A..S75R']
                        .sort_values(by=['arrival_time'], ascending=True)
                        #.merge(stops, on=['stop_id'], how='left')
                        ['stop_id']
                        .tolist()
                        )
a_rockaway_park

['A02S',
 'A03S',
 'A05S',
 'A06S',
 'A07S',
 'A09S',
 'A12S',
 'A15S',
 'A24S',
 'A27S',
 'A28S',
 'A31S',
 'A32S',
 'A34S',
 'A36S',
 'A38S',
 'A40S',
 'A41S',
 'A42S',
 'A46S',
 'A48S',
 'A51S',
 'A55S',
 'A57S',
 'A59S',
 'A60S',
 'A61S',
 'H02S',
 'H03S',
 'H04S',
 'H12S',
 'H13S',
 'H14S',
 'H15S']

In [12]:
tuid = "1739770740_C..S04R"
c_168_to_euclid = (
                        df.loc[df.trip_uid==tuid]
                        .sort_values(by=['arrival_time'], ascending=True)
                        .merge(stops, on=['stop_id'], how='left')
                        ['stop_id']
                        .tolist()
                        )

c_168_to_euclid

['A09S',
 'A10S',
 'A11S',
 'A12S',
 'A14S',
 'A15S',
 'A16S',
 'A17S',
 'A18S',
 'A19S',
 'A20S',
 'A21S',
 'A22S',
 'A24S',
 'A25S',
 'A27S',
 'A28S',
 'A30S',
 'A31S',
 'A32S',
 'A33S',
 'A34S',
 'A36S',
 'A38S',
 'A40S',
 'A41S',
 'A42S',
 'A43S',
 'A44S',
 'A45S',
 'A46S',
 'A47S',
 'A48S',
 'A49S',
 'A50S',
 'A51S',
 'A52S',
 'A53S',
 'A54S',
 'A55S']

In [13]:
tuid = "1739770380_E..S69R"
e_jamaica_wtc = (
                        df.loc[df.trip_uid==tuid]
                        .sort_values(by=['arrival_time'], ascending=True)
                        #.merge(stops, on=['stop_id'], how='left')
                        ['stop_id']
                        .tolist()
                        )
e_jamaica_wtc

['G05S',
 'G06S',
 'G07S',
 'F05S',
 'F06S',
 'F07S',
 'G08S',
 'G14S',
 'G21S',
 'F09S',
 'F11S',
 'F12S',
 'D14S',
 'A25S',
 'A27S',
 'A28S',
 'A30S',
 'A31S',
 'A32S',
 'A33S',
 'A34S',
 'E01S']

In [14]:
a_combined = sorted(set(a_far_rockaway + a_lefferts + a_rockaway_park))

a_combined

['A02S',
 'A03S',
 'A05S',
 'A06S',
 'A07S',
 'A09S',
 'A12S',
 'A15S',
 'A24S',
 'A27S',
 'A28S',
 'A31S',
 'A32S',
 'A34S',
 'A36S',
 'A38S',
 'A40S',
 'A41S',
 'A42S',
 'A46S',
 'A48S',
 'A51S',
 'A55S',
 'A57S',
 'A59S',
 'A60S',
 'A61S',
 'A63S',
 'A64S',
 'A65S',
 'H02S',
 'H03S',
 'H04S',
 'H06S',
 'H07S',
 'H08S',
 'H09S',
 'H10S',
 'H11S',
 'H12S',
 'H13S',
 'H14S',
 'H15S']

In [15]:
e_jamaica_wtc

['G05S',
 'G06S',
 'G07S',
 'F05S',
 'F06S',
 'F07S',
 'G08S',
 'G14S',
 'G21S',
 'F09S',
 'F11S',
 'F12S',
 'D14S',
 'A25S',
 'A27S',
 'A28S',
 'A30S',
 'A31S',
 'A32S',
 'A33S',
 'A34S',
 'E01S']

In [16]:
# raw sql data of actual train arrivals
arrivals = df[['trip_uid','stop_id','route_id','arrival_time','track']]

arrivals = arrivals.sort_values('arrival_time')

# 1 define the master topology
# create df of only the valid, scheduled route/stop combinations
valid_A_stops = a_combined
valid_C_stops = c_168_to_euclid
valid_E_stops = e_jamaica_wtc

valid_pairs = pd.DataFrame({
    'route_id':['A']*len(valid_A_stops) + ['C']*len(valid_C_stops) + ['E']*len(valid_E_stops),
    'stop_id': valid_A_stops + valid_C_stops + valid_E_stops
})

# filter raw data to only include these legist stops
arrivals = arrivals.merge(valid_pairs, on=['route_id','stop_id'], how='inner')


In [ ]:
# ── Encode station_order from the route sequence lists ───────────────
# Build a (route_id, stop_id) → position mapping from each route's
# ordered stop list. Position 0 = first southbound stop (northernmost).

# A train: use far_rockaway as the trunk, then append branch-only stops
station_order_A = {stop: i for i, stop in enumerate(a_combined)}
station_order_C = {stop: i for i, stop in enumerate(c_168_to_euclid)}
station_order_E = {stop: i for i, stop in enumerate(e_jamaica_wtc)}

# Merge into a single lookup: (route_id, stop_id) → position
station_order_map = {}
for stop, pos in station_order_A.items():
    station_order_map[('A', stop)] = pos
for stop, pos in station_order_C.items():
    station_order_map[('C', stop)] = pos
for stop, pos in station_order_E.items():
    station_order_map[('E', stop)] = pos

# Map into the DataFrame
arrivals['station_order'] = arrivals.apply(
    lambda r: station_order_map.get((r['route_id'], r['stop_id']), -1), axis=1
)

# Verify no unmatched stops
unmatched = (arrivals['station_order'] == -1).sum()
print(f"Unmatched (route, stop) pairs: {unmatched}")

# Sort by route, geographic station order, then arrival time
arrivals = arrivals.sort_values(['route_id', 'station_order', 'arrival_time']).reset_index(drop=True)

print(f"\nStation order examples:")
print(arrivals[['route_id','stop_id','station_order','arrival_time']].drop_duplicates(['route_id','stop_id']).to_string())

In [ ]:
# ── Compute target: minutes_until_next_train ─────────────────────────
# Sort chronologically within each (route, station) group
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)

# Forward diff: next arrival time minus current arrival time
arrivals['minutes_until_next_train'] = (
    arrivals
    .groupby(['route_id', 'stop_id'])['arrival_time']
    .shift(-1)
    .sub(arrivals['arrival_time'])
    .dt.total_seconds() / 60.0
)

# Last arrival per group has no next train → NaN
print(f"Rows before dropping NaN targets: {len(arrivals):,}")
print(f"NaN targets (last per group):     {arrivals['minutes_until_next_train'].isna().sum():,}")
arrivals = arrivals.dropna(subset=['minutes_until_next_train']).reset_index(drop=True)
print(f"Rows after dropping NaN targets:  {len(arrivals):,}")

# Filter out overnight gaps and glitches
HEADWAY_MAX = 25  # minutes
print(f"\nTargets > {HEADWAY_MAX} min: {(arrivals['minutes_until_next_train'] > HEADWAY_MAX).sum():,}")
arrivals = arrivals.loc[arrivals['minutes_until_next_train'] > 0].reset_index(drop=True)  # remove zero/negative
arrivals = arrivals.loc[arrivals['minutes_until_next_train'] <= HEADWAY_MAX].reset_index(drop=True)
print(f"Rows after filtering:             {len(arrivals):,}")

print(f"\nTarget summary:\n{arrivals['minutes_until_next_train'].describe()}")

In [ ]:
# ── Temporal features (NYC local time) ────────────────────────────────
import holidays

arrival_local = arrivals['arrival_time'].dt.tz_convert('America/New_York')
arrivals['hour'] = arrival_local.dt.hour
arrivals['day_of_week'] = arrival_local.dt.dayofweek  # 0=Mon, 6=Sun
arrivals['month'] = arrival_local.dt.month

# ── Cyclical encodings (sin/cos) ─────────────────────────────────────
# Daily periodicity (hour: period = 24)
arrivals['hour_sin'] = np.sin(2 * np.pi * arrivals['hour'] / 24)
arrivals['hour_cos'] = np.cos(2 * np.pi * arrivals['hour'] / 24)

# Weekly periodicity (day_of_week: period = 7)
arrivals['dow_sin'] = np.sin(2 * np.pi * arrivals['day_of_week'] / 7)
arrivals['dow_cos'] = np.cos(2 * np.pi * arrivals['day_of_week'] / 7)

# Monthly periodicity (month: period = 12)
arrivals['month_sin'] = np.sin(2 * np.pi * (arrivals['month'] - 1) / 12)
arrivals['month_cos'] = np.cos(2 * np.pi * (arrivals['month'] - 1) / 12)

# ── Binary flags ─────────────────────────────────────────────────────
arrivals['is_weekend'] = (arrivals['day_of_week'] >= 5).astype(int)

# US federal holidays via the `holidays` library
us_holidays = holidays.US(years=arrival_local.dt.year.unique().tolist())
arrivals['is_holiday'] = arrival_local.dt.date.map(lambda d: int(d in us_holidays))

# Summary
print(f"Cyclical features: hour_sin/cos, dow_sin/cos, month_sin/cos")
print(f"Binary flags: is_weekend={arrivals['is_weekend'].sum():,} rows,  is_holiday={arrivals['is_holiday'].sum():,} rows")
print(f"\nHolidays found in dataset:")
for d in sorted(set(arrival_local.dt.date[arrivals['is_holiday'] == 1])):
    print(f"  {d} — {us_holidays.get(d)}")

arrivals[['route_id','stop_id','arrival_time','hour','day_of_week','month',
          'hour_sin','hour_cos','dow_sin','dow_cos','is_weekend','is_holiday',
          'minutes_until_next_train']].head(10)

In [ ]:
# ── Rolling regime features on target (no leakage) ───────────────────
# shift(1) excludes the current row's target from the window.
# min_periods=1 so early rows in each group get partial-window stats
# rather than NaN.

arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)
grouped = arrivals.groupby(['route_id', 'stop_id'])['minutes_until_next_train']

# Window 3 — immediate signal
shifted = grouped.shift(1)
arrivals['rolling_mean_3']  = grouped.apply(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).values
arrivals['rolling_std_3']   = grouped.apply(lambda x: x.shift(1).rolling(3, min_periods=1).std()).values

# Window 5 — short-term trend
arrivals['rolling_mean_5']  = grouped.apply(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).values
arrivals['rolling_std_5']   = grouped.apply(lambda x: x.shift(1).rolling(5, min_periods=1).std()).values

# Window 10 — regime-level context
arrivals['rolling_mean_10'] = grouped.apply(lambda x: x.shift(1).rolling(10, min_periods=1).mean()).values
arrivals['rolling_max_10']  = grouped.apply(lambda x: x.shift(1).rolling(10, min_periods=1).max()).values

# Fill remaining NaN (first row per group has no prior observation)
for col in ['rolling_std_3', 'rolling_std_5']:
    arrivals[col] = arrivals[col].fillna(0)

print("Rolling features added:")
print(arrivals[['route_id','stop_id','minutes_until_next_train',
                'rolling_mean_3','rolling_std_3',
                'rolling_mean_5','rolling_std_5',
                'rolling_mean_10','rolling_max_10']].head(15))

print(f"\nNaN counts:")
for col in ['rolling_mean_3','rolling_std_3','rolling_mean_5','rolling_std_5','rolling_mean_10','rolling_max_10']:
    print(f"  {col}: {arrivals[col].isna().sum()}")

In [ ]:
# ── Fill remaining NaN in rolling features (first row per group) ─────
# Use per-group mean as fill so we don't leak cross-group info.
rolling_cols = ['rolling_mean_3', 'rolling_std_3',
                'rolling_mean_5', 'rolling_std_5',
                'rolling_mean_10', 'rolling_max_10']

for col in rolling_cols:
    group_means = arrivals.groupby(['route_id', 'stop_id'])[col].transform('mean')
    arrivals[col] = arrivals[col].fillna(group_means)

print("NaN counts after fill:")
for col in rolling_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nFinal shape: {arrivals.shape}")
print(f"Columns: {list(arrivals.columns)}")

In [ ]:
# ── group_id and time_idx (required by TFT) ──────────────────────────
# group_id: unique string identifier per time series (route_id + stop_id)
# time_idx:  sequential integer index within each group, ordered by arrival_time

arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)

arrivals['group_id'] = arrivals['route_id'] + '_' + arrivals['stop_id']
arrivals['time_idx'] = arrivals.groupby('group_id').cumcount()

print(f"Unique groups: {arrivals['group_id'].nunique()}")
print(f"time_idx range per group:")
print(arrivals.groupby('group_id')['time_idx'].agg(['min','max','count']).describe())
print(f"\nSample:")
print(arrivals[['group_id','time_idx','route_id','stop_id','arrival_time','minutes_until_next_train']].head(10))

In [ ]:
# ── Empirical median headway (known covariate) ───────────────────────
# Median headway by (route_id, is_weekend, hour), computed from TRAIN period only.
# This acts as a "what's normal right now" baseline the model can learn deviations from.

TRAIN_CUTOFF = pd.Timestamp('2025-10-29', tz='UTC')

train_mask = arrivals['arrival_time'] < TRAIN_CUTOFF
empirical_lookup = (
    arrivals.loc[train_mask]
    .groupby(['route_id', 'is_weekend', 'hour'])['minutes_until_next_train']
    .median()
    .rename('empirical_median')
)

arrivals = arrivals.merge(
    empirical_lookup,
    on=['route_id', 'is_weekend', 'hour'],
    how='left',
)

# Fill any (route, weekend, hour) combos that only appear in val/test with
# the global train median as fallback
global_median = arrivals.loc[train_mask, 'minutes_until_next_train'].median()
arrivals['empirical_median'] = arrivals['empirical_median'].fillna(global_median)

print(f'Empirical median lookup: {len(empirical_lookup)} entries')
print(f'NaN after merge (filled with global median {global_median:.2f}): '
      f'{arrivals["empirical_median"].isna().sum()}')
print(f'\nCorrelation with target: '
      f'{arrivals["empirical_median"].corr(arrivals["minutes_until_next_train"]):.4f}')
print(f'\nSample:')
print(arrivals[['route_id', 'is_weekend', 'hour', 'empirical_median',
                'minutes_until_next_train']].sample(10, random_state=42))

In [ ]:
# ── Group 5: Deviation features (composites of rolling + empirical_median) ──
# These make regime deviations explicit so the model doesn't have to learn
# the arithmetic internally.  All inputs already exist; no cross-group joins.

# 1. headway_deviation_ratio: rolling_mean_3 / empirical_median
#    >1 = headways worse than normal, <1 = better than normal
arrivals['headway_deviation_ratio'] = arrivals['rolling_mean_3'] / arrivals['empirical_median']

# 2. headway_deviation_signed: rolling_mean_3 - empirical_median
#    Absolute deviation in minutes (complementary to ratio)
arrivals['headway_deviation_signed'] = arrivals['rolling_mean_3'] - arrivals['empirical_median']

# 3. deviation_trend: rolling_mean_3 - rolling_mean_10
#    Positive = short-term worse than medium-term (deteriorating)
#    Negative = short-term better than medium-term (recovering)
arrivals['deviation_trend'] = arrivals['rolling_mean_3'] - arrivals['rolling_mean_10']

# 4. deviation_streak: count of the last 5 headways (shifted by 1) that
#    exceeded the empirical_median for that row.  High streak = sustained
#    disruption; low streak = isolated blip.
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)

above_median = (arrivals['minutes_until_next_train'] > arrivals['empirical_median']).astype(int)
arrivals['deviation_streak'] = (
    above_median
    .groupby([arrivals['route_id'], arrivals['stop_id']])
    .apply(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .values
)

# Fill NaN (first row per group where shift produces NaN)
arrivals['deviation_streak'] = arrivals['deviation_streak'].fillna(0)

# ── Summary ──────────────────────────────────────────────────────────
deviation_cols = ['headway_deviation_ratio', 'headway_deviation_signed',
                  'deviation_trend', 'deviation_streak']

print("Group 5 — Deviation features added:")
print(arrivals[deviation_cols].describe().round(3))

print(f"\nNaN counts:")
for col in deviation_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nCorrelations with target (minutes_until_next_train):")
for col in deviation_cols:
    corr = arrivals[col].corr(arrivals['minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

In [ ]:
# ── Group 3: Time Since Last Upstream Departure ──────────────────────
# For each arrival at station k, compute how long ago the last same-route
# train passed station k-1 (and k-2).  Uses pd.merge_asof for efficient
# backward time lookup.  Strict < (allow_exact_matches=False) to prevent
# leakage.

# ── Step 1: Build upstream stop mapping from branch-specific route lists
def _build_upstream(stop_list, route_id, max_offset=2):
    """Map (route, stop, offset) → upstream stop_id."""
    m = {}
    for i, stop in enumerate(stop_list):
        for off in range(1, max_offset + 1):
            if i - off >= 0:
                m[(route_id, stop, off)] = stop_list[i - off]
    return m

upstream_map = {}
for branch in [a_far_rockaway, a_lefferts, a_rockaway_park]:
    upstream_map.update(_build_upstream(branch, 'A'))
upstream_map.update(_build_upstream(c_168_to_euclid, 'C'))
upstream_map.update(_build_upstream(e_jamaica_wtc, 'E'))

# Flatten to offset-specific dicts for fast vectorized lookup
_us1 = {(r, s): v for (r, s, o), v in upstream_map.items() if o == 1}
_us2 = {(r, s): v for (r, s, o), v in upstream_map.items() if o == 2}

_key = list(zip(arrivals['route_id'], arrivals['stop_id']))
arrivals['_upstream_stop_1'] = [_us1.get(k) for k in _key]
arrivals['_upstream_stop_2'] = [_us2.get(k) for k in _key]

print(f"Upstream mappings: {len(_us1)} stops with k-1, {len(_us2)} stops with k-2")
print(f"Rows with upstream_stop_1: {arrivals['_upstream_stop_1'].notna().sum():,} / {len(arrivals):,}")
print(f"Rows with upstream_stop_2: {arrivals['_upstream_stop_2'].notna().sum():,} / {len(arrivals):,}")

# ── Step 2: merge_asof to find most recent upstream arrival ──────────
# Reference table: all arrivals as potential upstream matches
ref = arrivals[['route_id', 'stop_id', 'arrival_time']].copy()
ref = ref.rename(columns={'stop_id': '_merge_stop'})
ref['_upstream_time'] = ref['arrival_time']       # preserve for time-diff
ref = ref.sort_values('arrival_time')

arrivals = arrivals.sort_values('arrival_time').reset_index(drop=True)

for us_col, out_col in [
    ('_upstream_stop_1', 'time_since_upstream_1'),
    ('_upstream_stop_2', 'time_since_upstream_2'),
]:
    # Left: current arrivals keyed by their upstream stop
    left = arrivals[['route_id', us_col, 'arrival_time']].copy()
    left = left.rename(columns={us_col: '_merge_stop'})
    left['_orig_idx'] = left.index

    has_upstream = left['_merge_stop'].notna()
    left_valid = left.loc[has_upstream].sort_values('arrival_time')

    # merge_asof: find most recent upstream arrival strictly before
    merged = pd.merge_asof(
        left_valid[['_orig_idx', 'route_id', '_merge_stop', 'arrival_time']],
        ref[['route_id', '_merge_stop', 'arrival_time', '_upstream_time']],
        on='arrival_time',
        by=['route_id', '_merge_stop'],
        direction='backward',
        allow_exact_matches=False,
    )

    # Time since upstream in minutes, clipped to HEADWAY_MAX
    merged[out_col] = (
        (merged['arrival_time'] - merged['_upstream_time']).dt.total_seconds() / 25.0
    )
    merged[out_col] = merged[out_col].clip(upper=HEADWAY_MAX)

    # Map back to original index
    arrivals[out_col] = np.nan
    arrivals.loc[merged['_orig_idx'].values, out_col] = merged[out_col].values

    n_valid = arrivals[out_col].notna().sum()
    n_nan = arrivals[out_col].isna().sum()
    print(f"\n{out_col}: {n_valid:,} valid, {n_nan:,} NaN")

# ── Step 3: Fill NaN with empirical_median ───────────────────────────
# First stations per route have no upstream → NaN
# Sparse late-night gaps may also produce NaN
for col in ['time_since_upstream_1', 'time_since_upstream_2']:
    arrivals[col] = arrivals[col].fillna(arrivals['empirical_median'])

# Re-sort to standard order and clean up temp columns
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)
arrivals = arrivals.drop(columns=['_upstream_stop_1', '_upstream_stop_2'])

# ── Summary ──────────────────────────────────────────────────────────
upstream_cols = ['time_since_upstream_1', 'time_since_upstream_2']
print("\n── Group 3 — Time Since Upstream summary ──")
print(arrivals[upstream_cols].describe().round(3))

print(f"\nNaN counts:")
for col in upstream_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nCorrelations with target (minutes_until_next_train):")
for col in upstream_cols:
    corr = arrivals[col].corr(arrivals['minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

In [ ]:
# ── Group 1: Same-Route Upstream Headway ─────────────────────────────
# For each arrival at station k, look up the most recent *completed*
# headway at upstream stations k-1, k-2, k-3 on the same route.
# Uses pd.merge_asof (backward, strict <) to prevent leakage.

# ── Step 1: Extend upstream mapping to k-3 ───────────────────────────
def _build_upstream_ext(stop_list, route_id, max_offset=3):
    """Map (route, stop, offset) → upstream stop_id."""
    m = {}
    for i, stop in enumerate(stop_list):
        for off in range(1, max_offset + 1):
            if i - off >= 0:
                m[(route_id, stop, off)] = stop_list[i - off]
    return m

upstream_map_ext = {}
for branch in [a_far_rockaway, a_lefferts, a_rockaway_park]:
    upstream_map_ext.update(_build_upstream_ext(branch, 'A'))
upstream_map_ext.update(_build_upstream_ext(c_168_to_euclid, 'C'))
upstream_map_ext.update(_build_upstream_ext(e_jamaica_wtc, 'E'))

_us1 = {(r, s): v for (r, s, o), v in upstream_map_ext.items() if o == 1}
_us2 = {(r, s): v for (r, s, o), v in upstream_map_ext.items() if o == 2}
_us3 = {(r, s): v for (r, s, o), v in upstream_map_ext.items() if o == 3}

_key = list(zip(arrivals['route_id'], arrivals['stop_id']))
arrivals['_us_stop_1'] = [_us1.get(k) for k in _key]
arrivals['_us_stop_2'] = [_us2.get(k) for k in _key]
arrivals['_us_stop_3'] = [_us3.get(k) for k in _key]

print(f"Upstream k-1: {len(_us1)}, k-2: {len(_us2)}, k-3: {len(_us3)} stop mappings")

# ── Step 2: Build reference table of completed headways per station ──
# Each row is a (route, stop, arrival_time, headway) — the headway that
# was finalized at that arrival_time.
ref_hw = arrivals[['route_id', 'stop_id', 'arrival_time', 'minutes_until_next_train']].copy()
ref_hw = ref_hw.rename(columns={
    'stop_id': '_merge_stop',
    'minutes_until_next_train': '_upstream_hw',
})
ref_hw = ref_hw.sort_values('arrival_time')

arrivals = arrivals.sort_values('arrival_time').reset_index(drop=True)

# ── Step 3: merge_asof for each offset ───────────────────────────────
for us_col, out_col in [
    ('_us_stop_1', 'upstream_headway_1'),
    ('_us_stop_2', 'upstream_headway_2'),
    ('_us_stop_3', 'upstream_headway_3'),
]:
    left = arrivals[['route_id', us_col, 'arrival_time']].copy()
    left = left.rename(columns={us_col: '_merge_stop'})
    left['_orig_idx'] = left.index

    has_upstream = left['_merge_stop'].notna()
    left_valid = left.loc[has_upstream].sort_values('arrival_time')

    merged = pd.merge_asof(
        left_valid[['_orig_idx', 'route_id', '_merge_stop', 'arrival_time']],
        ref_hw[['route_id', '_merge_stop', 'arrival_time', '_upstream_hw']],
        on='arrival_time',
        by=['route_id', '_merge_stop'],
        direction='backward',
        allow_exact_matches=False,
    )

    arrivals[out_col] = np.nan
    arrivals.loc[merged['_orig_idx'].values, out_col] = merged['_upstream_hw'].values

    n_valid = arrivals[out_col].notna().sum()
    n_nan = arrivals[out_col].isna().sum()
    print(f"{out_col}: {n_valid:,} valid, {n_nan:,} NaN")

# ── Step 4: Fill NaN with empirical_median, compute upstream_deviation_ratio
for col in ['upstream_headway_1', 'upstream_headway_2', 'upstream_headway_3']:
    arrivals[col] = arrivals[col].fillna(arrivals['empirical_median'])

# upstream_deviation_ratio: upstream_headway_1 / empirical_median
# (deferred from Group 5 since it depends on upstream_headway_1)
arrivals['upstream_deviation_ratio'] = arrivals['upstream_headway_1'] / arrivals['empirical_median']

# Re-sort and clean up
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)
arrivals = arrivals.drop(columns=['_us_stop_1', '_us_stop_2', '_us_stop_3'])

# ── Summary ──────────────────────────────────────────────────────────
g1_cols = ['upstream_headway_1', 'upstream_headway_2', 'upstream_headway_3',
           'upstream_deviation_ratio']
print("\n── Group 1 — Upstream Headway summary ──")
print(arrivals[g1_cols].describe().round(3))

print(f"\nNaN counts:")
for col in g1_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nCorrelations with target (minutes_until_next_train):")
for col in g1_cols:
    corr = arrivals[col].corr(arrivals['minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

In [ ]:
# ── Group 2: Upstream Travel Time (via trip_uid) ─────────────────────
# For each arrival at station k, compute:
#   - last_train_travel_time_1: how long the *same physical train* (trip_uid)
#     took to travel from station k-1 to k (minutes).
#   - travel_time_deviation_1: ratio of actual travel time to the median
#     travel time for that segment, normalized by (route, stop, is_weekend, hour).
#   - last_train_travel_time_2: same but from k-2 to k.
#
# This requires a self-join on trip_uid to find the same train at upstream stops.

# ── Step 1: Rebuild upstream stop mapping (k-1, k-2) ────────────────
def _build_upstream_g2(stop_list, route_id, max_offset=2):
    """Map (route, stop, offset) → upstream stop_id."""
    m = {}
    for i, stop in enumerate(stop_list):
        for off in range(1, max_offset + 1):
            if i - off >= 0:
                m[(route_id, stop, off)] = stop_list[i - off]
    return m

_us_map = {}
for branch in [a_far_rockaway, a_lefferts, a_rockaway_park]:
    _us_map.update(_build_upstream_g2(branch, 'A'))
_us_map.update(_build_upstream_g2(c_168_to_euclid, 'C'))
_us_map.update(_build_upstream_g2(e_jamaica_wtc, 'E'))

_us1 = {(r, s): v for (r, s, o), v in _us_map.items() if o == 1}
_us2 = {(r, s): v for (r, s, o), v in _us_map.items() if o == 2}

# ── Step 2: Build trip_uid arrival lookup ────────────────────────────
# (trip_uid, stop_id) → arrival_time
trip_stop_time = arrivals.set_index(['trip_uid', 'stop_id'])['arrival_time']
trip_stop_time = trip_stop_time[~trip_stop_time.index.duplicated(keep='first')]

print(f"trip_stop_time lookup: {len(trip_stop_time):,} entries")

# ── Step 3: Look up same trip_uid at upstream stops ──────────────────
_key = list(zip(arrivals['route_id'], arrivals['stop_id']))
us_stop_1 = [_us1.get(k) for k in _key]
us_stop_2 = [_us2.get(k) for k in _key]

trip_uids = arrivals['trip_uid'].values

# Vectorized lookup using reindex
_MISS = '__MISSING__'
idx_1 = pd.MultiIndex.from_tuples(
    [(t, s) if s is not None else (t, _MISS) for t, s in zip(trip_uids, us_stop_1)],
    names=['trip_uid', 'stop_id']
)
idx_2 = pd.MultiIndex.from_tuples(
    [(t, s) if s is not None else (t, _MISS) for t, s in zip(trip_uids, us_stop_2)],
    names=['trip_uid', 'stop_id']
)

# Reindex and reset index to align with arrivals (preserve tz-aware dtype)
upstream_arrival_1 = trip_stop_time.reindex(idx_1)
upstream_arrival_1.index = arrivals.index

upstream_arrival_2 = trip_stop_time.reindex(idx_2)
upstream_arrival_2.index = arrivals.index

# Compute travel times in minutes
arrivals['last_train_travel_time_1'] = (
    (arrivals['arrival_time'] - upstream_arrival_1)
    .dt.total_seconds() / 60.0
)
arrivals['last_train_travel_time_2'] = (
    (arrivals['arrival_time'] - upstream_arrival_2)
    .dt.total_seconds() / 60.0
)

# Sanity: negative or zero travel times → NaN
arrivals.loc[arrivals['last_train_travel_time_1'] <= 0, 'last_train_travel_time_1'] = np.nan
arrivals.loc[arrivals['last_train_travel_time_2'] <= 0, 'last_train_travel_time_2'] = np.nan

# Cap extreme travel times (> 30 min between adjacent stops is data error)
MAX_SEGMENT_TIME = 30.0
arrivals.loc[arrivals['last_train_travel_time_1'] > MAX_SEGMENT_TIME, 'last_train_travel_time_1'] = np.nan
arrivals.loc[arrivals['last_train_travel_time_2'] > MAX_SEGMENT_TIME * 2, 'last_train_travel_time_2'] = np.nan

n1_valid = arrivals['last_train_travel_time_1'].notna().sum()
n2_valid = arrivals['last_train_travel_time_2'].notna().sum()
print(f"\nlast_train_travel_time_1: {n1_valid:,} valid, {len(arrivals)-n1_valid:,} NaN")
print(f"last_train_travel_time_2: {n2_valid:,} valid, {len(arrivals)-n2_valid:,} NaN")

# ── Step 4: Segment median travel times (training period only) ───────
train_mask = arrivals['arrival_time'] < TRAIN_CUTOFF

segment_median_1 = (
    arrivals.loc[train_mask & arrivals['last_train_travel_time_1'].notna()]
    .groupby(['route_id', 'stop_id', 'is_weekend', 'hour'])['last_train_travel_time_1']
    .median()
    .rename('_seg_median_1')
)

segment_median_2 = (
    arrivals.loc[train_mask & arrivals['last_train_travel_time_2'].notna()]
    .groupby(['route_id', 'stop_id', 'is_weekend', 'hour'])['last_train_travel_time_2']
    .median()
    .rename('_seg_median_2')
)

print(f"\nSegment median lookup: {len(segment_median_1):,} entries (k-1), "
      f"{len(segment_median_2):,} entries (k-2)")

# Merge segment medians
arrivals = arrivals.merge(segment_median_1, on=['route_id', 'stop_id', 'is_weekend', 'hour'], how='left')
arrivals = arrivals.merge(segment_median_2, on=['route_id', 'stop_id', 'is_weekend', 'hour'], how='left')

# ── Step 5: Deviation ratio (capped at 5.0 to tame outliers) ─────────
arrivals['travel_time_deviation_1'] = (
    (arrivals['last_train_travel_time_1'] / arrivals['_seg_median_1']).clip(upper=5.0)
)

# ── Step 6: Fill NaN ────────────────────────────────────────────────
# Absolute: fill with segment median, then global median as fallback
global_seg_median_1 = arrivals.loc[train_mask, 'last_train_travel_time_1'].median()
global_seg_median_2 = arrivals.loc[train_mask, 'last_train_travel_time_2'].median()

arrivals['last_train_travel_time_1'] = (
    arrivals['last_train_travel_time_1']
    .fillna(arrivals['_seg_median_1'])
    .fillna(global_seg_median_1)
)
arrivals['last_train_travel_time_2'] = (
    arrivals['last_train_travel_time_2']
    .fillna(arrivals['_seg_median_2'])
    .fillna(global_seg_median_2)
)

# Deviation ratio: fill NaN with 1.0 (normal speed)
arrivals['travel_time_deviation_1'] = arrivals['travel_time_deviation_1'].fillna(1.0)

# Clean up temp columns
arrivals = arrivals.drop(columns=['_seg_median_1', '_seg_median_2'])

# ── Summary ──────────────────────────────────────────────────────────
g2_cols = ['last_train_travel_time_1', 'travel_time_deviation_1', 'last_train_travel_time_2']

print("\n── Group 2 — Upstream Travel Time summary ──")
print(arrivals[g2_cols].describe().round(3))

print(f"\nNaN counts:")
for col in g2_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nCorrelations with target (minutes_until_next_train):")
for col in g2_cols:
    corr = arrivals[col].corr(arrivals['minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

print(f"\nFill values: k-1 global median={global_seg_median_1:.2f} min, "
      f"k-2 global median={global_seg_median_2:.2f} min")

In [ ]:
# ── Group 4: Cross-Route Upstream Signals (Shared Trunk) ─────────────
# On shared A/C/E trunk stops, riders care about *any* train, not just
# the same route.  These features capture cross-route supply at shared
# stations.  Non-shared stops get same-route fallback values.
#
# Features:
#   any_route_headway_upstream    — most recent headway at k-1 across ALL routes
#   any_route_time_since_upstream — time since ANY A/C/E train at k-1
#   cross_route_trains_last_10min — count of A/C/E trains at k-1 in last 10 min

# ── Step 1: Define shared stops ──────────────────────────────────────
a_set = set(a_combined)
c_set = set(c_168_to_euclid)
e_set = set(e_jamaica_wtc)

shared_AC  = a_set & c_set
shared_CE  = c_set & e_set
shared_ACE = a_set & c_set & e_set
shared_any = a_set | c_set | e_set  # all stops served by any route

# All stops where at least 2 routes serve the same platform
shared_stops = (a_set & c_set) | (c_set & e_set) | (a_set & e_set)

print(f"Shared stops (A∩C): {len(shared_AC)} — {sorted(shared_AC)[:5]}...")
print(f"Shared stops (C∩E): {len(shared_CE)}")
print(f"Shared stops (A∩C∩E): {len(shared_ACE)}")
print(f"Total shared (≥2 routes): {len(shared_stops)}")

# Build upstream stop for each (route, stop)
def _build_upstream_shared(stop_list, route_id):
    m = {}
    for i, stop in enumerate(stop_list):
        if i > 0:
            m[(route_id, stop)] = stop_list[i - 1]
    return m

_us_shared = {}
for branch in [a_far_rockaway, a_lefferts, a_rockaway_park]:
    _us_shared.update(_build_upstream_shared(branch, 'A'))
_us_shared.update(_build_upstream_shared(c_168_to_euclid, 'C'))
_us_shared.update(_build_upstream_shared(e_jamaica_wtc, 'E'))

_key = list(zip(arrivals['route_id'], arrivals['stop_id']))
arrivals['_xr_upstream'] = [_us_shared.get(k) for k in _key]

arrivals['_is_shared'] = arrivals['stop_id'].isin(shared_stops).astype(int)
n_shared = arrivals['_is_shared'].sum()
print(f"\nRows at shared stops: {n_shared:,} / {len(arrivals):,} ({100*n_shared/len(arrivals):.1f}%)")

# ── Step 2: Cross-route reference table (ALL routes at each stop) ────
ref_xr = arrivals[['stop_id', 'arrival_time', 'minutes_until_next_train']].copy()
ref_xr = ref_xr.rename(columns={
    'stop_id': '_xr_merge_stop',
    'minutes_until_next_train': '_xr_hw',
})
ref_xr['_xr_time'] = ref_xr['arrival_time']
ref_xr = ref_xr.sort_values('arrival_time')

arrivals = arrivals.sort_values('arrival_time').reset_index(drop=True)

# ── Step 3: any_route_headway_upstream via merge_asof ────────────────
left = arrivals[['_xr_upstream', 'arrival_time']].copy()
left = left.rename(columns={'_xr_upstream': '_xr_merge_stop'})
left['_orig_idx'] = left.index

has_upstream = left['_xr_merge_stop'].notna()
left_valid = left.loc[has_upstream].sort_values('arrival_time')

merged_xr = pd.merge_asof(
    left_valid[['_orig_idx', '_xr_merge_stop', 'arrival_time']],
    ref_xr[['_xr_merge_stop', 'arrival_time', '_xr_hw', '_xr_time']],
    on='arrival_time',
    by='_xr_merge_stop',
    direction='backward',
    allow_exact_matches=False,
)

arrivals['any_route_headway_upstream'] = np.nan
arrivals.loc[merged_xr['_orig_idx'].values, 'any_route_headway_upstream'] = merged_xr['_xr_hw'].values

arrivals['any_route_time_since_upstream'] = np.nan
time_diff = (merged_xr['arrival_time'] - merged_xr['_xr_time']).dt.total_seconds() / 60.0
arrivals.loc[merged_xr['_orig_idx'].values, 'any_route_time_since_upstream'] = time_diff.values

arrivals['any_route_time_since_upstream'] = arrivals['any_route_time_since_upstream'].clip(upper=HEADWAY_MAX)

n1 = arrivals['any_route_headway_upstream'].notna().sum()
n2 = arrivals['any_route_time_since_upstream'].notna().sum()
print(f"\nany_route_headway_upstream: {n1:,} valid, {len(arrivals)-n1:,} NaN")
print(f"any_route_time_since_upstream: {n2:,} valid, {len(arrivals)-n2:,} NaN")

# ── Step 4: cross_route_trains_last_10min ────────────────────────────
# Count trains (any route) at the upstream stop in the last 10 minutes.
# Use float64 seconds since epoch (resolution-agnostic — avoids the
# ns-vs-µs mismatch that caused all-zero counts in the previous version).
# Vectorized per upstream stop via np.searchsorted on arrays.

_epoch = pd.Timestamp('1970-01-01', tz='UTC')

# Build per-stop sorted arrival-second arrays from ref_xr
ref_xr['_sec'] = (ref_xr['arrival_time'] - _epoch).dt.total_seconds()
stop_arrival_sec = {}
for stop_id, grp in ref_xr.groupby('_xr_merge_stop'):
    stop_arrival_sec[stop_id] = grp['_sec'].sort_values().values

# Arrival seconds for the main DataFrame
arrivals['_arr_sec'] = (arrivals['arrival_time'] - _epoch).dt.total_seconds()

TEN_MIN_SEC = 600.0  # 10 minutes in seconds

counts = np.full(len(arrivals), np.nan, dtype=np.float32)

# Vectorized searchsorted per upstream stop (~100 stops instead of 4M rows)
for us_stop, ref_arr in stop_arrival_sec.items():
    mask = (arrivals['_xr_upstream'] == us_stop).values
    if not mask.any():
        continue
    t = arrivals.loc[mask, '_arr_sec'].values
    right = np.searchsorted(ref_arr, t, side='left')          # first >= t
    left_idx = np.searchsorted(ref_arr, t - TEN_MIN_SEC, side='right')  # first > t-10min
    counts[mask] = (right - left_idx).astype(np.float32)

arrivals['cross_route_trains_last_10min'] = counts

# Clean up temp column
arrivals = arrivals.drop(columns=['_arr_sec'])

n3 = arrivals['cross_route_trains_last_10min'].notna().sum()
print(f"cross_route_trains_last_10min: {n3:,} valid, {len(arrivals)-n3:,} NaN")

# ── Step 5: Fill NaN ────────────────────────────────────────────────
non_shared = ~arrivals['stop_id'].isin(shared_stops)

arrivals['any_route_headway_upstream'] = arrivals['any_route_headway_upstream'].fillna(
    arrivals['empirical_median']
)
arrivals['any_route_time_since_upstream'] = arrivals['any_route_time_since_upstream'].fillna(
    arrivals['empirical_median']
)
arrivals['cross_route_trains_last_10min'] = arrivals['cross_route_trains_last_10min'].fillna(0)

# Clean up temp columns
arrivals = arrivals.drop(columns=['_xr_upstream', '_is_shared'])
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)

# ── Summary ──────────────────────────────────────────────────────────
g4_cols = ['any_route_headway_upstream', 'any_route_time_since_upstream',
           'cross_route_trains_last_10min']

print("\n── Group 4 — Cross-Route Signals summary ──")
print(arrivals[g4_cols].describe().round(3))

print(f"\nNaN counts:")
for col in g4_cols:
    print(f"  {col}: {arrivals[col].isna().sum()}")

print(f"\nCorrelations with target (minutes_until_next_train):")
for col in g4_cols:
    corr = arrivals[col].corr(arrivals['minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

# Break out correlations for shared vs non-shared stops
shared_mask = arrivals['stop_id'].isin(shared_stops)
print(f"\nCorrelations at SHARED stops only ({shared_mask.sum():,} rows):")
for col in g4_cols:
    corr = arrivals.loc[shared_mask, col].corr(arrivals.loc[shared_mask, 'minutes_until_next_train'])
    print(f"  {col}: {corr:.4f}")

In [ ]:
# ── Track feature: clean & bin rare values ────────────────────────────
# track is a categorical feature indicating express/local track assignment.
# Reroutes (unusual track for a stop) signal disruption.
# 4 dominant values (A1=45%, A3=30%, D3=11%, K1=7.5%) cover 95%+ of rows.
# Rare values (< 100 occurrences) are binned into 'OTHER'.

# Fill NaN (3,215 rows) with per-(route, stop) mode
track_mode = (
    arrivals.groupby(['route_id', 'stop_id'])['track']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'OTHER')
)
arrivals['track'] = arrivals['track'].fillna(
    arrivals.set_index(['route_id', 'stop_id']).index.map(track_mode)
)

# Bin rare tracks (< 100 occurrences) into 'OTHER'
track_counts = arrivals['track'].value_counts()
rare_tracks = track_counts[track_counts < 100].index.tolist()
arrivals.loc[arrivals['track'].isin(rare_tracks), 'track'] = 'OTHER'

# Fill any remaining NaN
arrivals['track'] = arrivals['track'].fillna('OTHER')

print("── Track feature summary ──")
print(f"NaN count: {arrivals['track'].isna().sum()}")
print(f"\nValue counts:")
print(arrivals['track'].value_counts())
print(f"\nCorrelation via one-hot (top 4 tracks vs target):")
for t in arrivals['track'].value_counts().head(4).index:
    is_track = (arrivals['track'] == t).astype(int)
    corr = is_track.corr(arrivals['minutes_until_next_train'])
    print(f"  track={t}: {corr:.4f}")

In [ ]:
# ── Active trip count: trains running on the same route in the last 30 min ──
# Count of distinct trip_uids on the same route that had an arrival *anywhere*
# in the 30 minutes before the current arrival.  Fewer active trains = longer
# expected headways.  Vectorized per route via np.searchsorted.

_epoch = pd.Timestamp('1970-01-01', tz='UTC')
THIRTY_MIN_SEC = 1800.0

arrivals = arrivals.sort_values('arrival_time').reset_index(drop=True)
arrivals['_arr_sec'] = (arrivals['arrival_time'] - _epoch).dt.total_seconds()

active_counts = np.zeros(len(arrivals), dtype=np.float32)

for route in ['A', 'C', 'E']:
    route_mask = (arrivals['route_id'] == route).values
    route_df = arrivals.loc[route_mask].copy()

    # Build per-trip_uid arrival times: for each trip, take the LAST arrival
    # on this route so we know the trip was "active" up to that point
    route_arrivals_sec = route_df['_arr_sec'].values  # already sorted by time

    # For counting unique trips, we need a different approach:
    # For each arrival at time t, count distinct trip_uids with an arrival
    # in [t - 30min, t).  Use a sliding window on sorted data.
    trip_uids_arr = route_df['trip_uid'].values
    times = route_df['_arr_sec'].values
    n = len(times)

    # Sliding window: maintain set of active trips
    left_ptr = 0
    active_trips = {}  # trip_uid -> count of arrivals in window

    route_counts = np.zeros(n, dtype=np.float32)
    for i in range(n):
        t = times[i]
        # Advance left pointer to remove trips outside window
        while left_ptr < i and times[left_ptr] <= t - THIRTY_MIN_SEC:
            tid = trip_uids_arr[left_ptr]
            active_trips[tid] -= 1
            if active_trips[tid] == 0:
                del active_trips[tid]
            left_ptr += 1
        # Current arrival is NOT included (strict <) to avoid leakage
        route_counts[i] = len(active_trips)
        # Add current arrival to window for future rows
        tid = trip_uids_arr[i]
        active_trips[tid] = active_trips.get(tid, 0) + 1

    active_counts[route_mask] = route_counts

arrivals['active_trip_count'] = active_counts
arrivals = arrivals.drop(columns=['_arr_sec'])

# Re-sort to standard order
arrivals = arrivals.sort_values(['route_id', 'stop_id', 'arrival_time']).reset_index(drop=True)

# ── Summary ──────────────────────────────────────────────────────────
print("── Active Trip Count summary ──")
print(arrivals['active_trip_count'].describe().round(2))
print(f"\nNaN count: {arrivals['active_trip_count'].isna().sum()}")
print(f"\nCorrelation with target: {arrivals['active_trip_count'].corr(arrivals['minutes_until_next_train']):.4f}")
print(f"\nBreakdown by route:")
for route in ['A', 'C', 'E']:
    mask = arrivals['route_id'] == route
    corr = arrivals.loc[mask, 'active_trip_count'].corr(arrivals.loc[mask, 'minutes_until_next_train'])
    mean = arrivals.loc[mask, 'active_trip_count'].mean()
    print(f"  {route}: mean={mean:.1f} active trips, corr={corr:.4f}")

In [ ]:
# ── Autocorrelation analysis to determine lookback window ─────────────
from statsmodels.tsa.stattools import acf, pacf
import matplotlib.pyplot as plt

MAX_LAGS = 60  # examine up to 60 steps back

# Compute ACF/PACF per group, then average across groups for a robust signal.
# Sample a representative set of groups (one per route + a few high-volume ones).
sample_groups = (
    arrivals.groupby('group_id')['time_idx']
    .count()
    .sort_values(ascending=False)
    .head(12)
    .index.tolist()
)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# ── 1. Per-group ACF overlay ─────────────────────────────────────────
ax = axes[0]
for gid in sample_groups:
    series = arrivals.loc[arrivals['group_id'] == gid, 'minutes_until_next_train'].values
    acf_vals = acf(series, nlags=MAX_LAGS, fft=True)
    ax.plot(range(MAX_LAGS + 1), acf_vals, alpha=0.4, label=gid)
ax.axhline(0, color='k', linewidth=0.5)
ax.axhline(0.05, color='grey', linestyle='--', linewidth=0.5)
ax.set_title('ACF per group (top 12 by volume)')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.legend(fontsize=7, ncol=3, loc='upper right')

# ── 2. Average ACF across ALL groups ─────────────────────────────────
ax = axes[1]
all_acfs = []
for gid, grp in arrivals.groupby('group_id'):
    series = grp['minutes_until_next_train'].values
    if len(series) > MAX_LAGS + 1:
        all_acfs.append(acf(series, nlags=MAX_LAGS, fft=True))

avg_acf = np.mean(all_acfs, axis=0)
ax.bar(range(MAX_LAGS + 1), avg_acf, color='steelblue', width=0.8)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_title(f'Average ACF across {len(all_acfs)} groups')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')

# Annotate where ACF drops below thresholds
for thresh in [0.1, 0.05]:
    below = np.where(avg_acf[1:] < thresh)[0]
    if len(below) > 0:
        lag_at = below[0] + 1
        ax.axvline(lag_at, color='red' if thresh == 0.05 else 'orange',
                    linestyle='--', linewidth=1, label=f'ACF < {thresh} at lag {lag_at}')
ax.legend()

# ── 3. Average PACF across ALL groups ────────────────────────────────
ax = axes[2]
all_pacfs = []
for gid, grp in arrivals.groupby('group_id'):
    series = grp['minutes_until_next_train'].values
    if len(series) > MAX_LAGS + 1:
        try:
            all_pacfs.append(pacf(series, nlags=MAX_LAGS))
        except Exception:
            pass

avg_pacf = np.mean(all_pacfs, axis=0)
ax.bar(range(MAX_LAGS + 1), avg_pacf, color='darkorange', width=0.8)
ax.axhline(0, color='k', linewidth=0.5)
ax.axhline(0.05, color='grey', linestyle='--', linewidth=0.5)
ax.axhline(-0.05, color='grey', linestyle='--', linewidth=0.5)
ax.set_title(f'Average PACF across {len(all_pacfs)} groups')
ax.set_xlabel('Lag')
ax.set_ylabel('Partial Autocorrelation')

plt.tight_layout()
plt.show()

# ── Summary stats ────────────────────────────────────────────────────
print("\n── Lookback window guidance ──")
print(f"Average ACF values at key lags:")
for lag in [1, 3, 5, 10, 15, 20, 30, 40, 50, 60]:
    if lag <= MAX_LAGS:
        print(f"  lag {lag:3d}: ACF={avg_acf[lag]:.4f}  PACF={avg_pacf[lag]:.4f}")

# Find where average ACF drops below thresholds
for thresh_name, thresh in [("0.10", 0.10), ("0.05", 0.05), ("0.02", 0.02)]:
    below = np.where(avg_acf[1:] < float(thresh_name))[0]
    if len(below) > 0:
        print(f"  Average ACF < {thresh_name} at lag {below[0] + 1}")
    else:
        print(f"  Average ACF never drops below {thresh_name} within {MAX_LAGS} lags")

In [ ]:
# ── Save to parquet ──────────────────────────────────────────────────
import os

out_dir = '../local_artifacts/processed_data'
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, 'tft_training_data.parquet')
arrivals.to_parquet(out_path, index=False, engine='pyarrow')

print(f"Saved {len(arrivals):,} rows → {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1e6:.1f} MB")